In [2]:
import pandas as pd
import numpy as np
import glob
import os

print("All imports successful!")

All imports successful!


Creating master dataset

In [3]:
import pandas as pd

dfs = []

for year in range(2000, 2011):
    print(f"Processing {year}")

    file_path = (
        f"C:\\Users\\Niraj Mhatre\\projects\\Mortgage-Portfolio-Risk-Analytics-and-IFRS-9-Provisioning-Framework"
        f"\\Data\\sample_{year}\\sample_orig_{year}.txt"
    )

    df = pd.read_csv(
        file_path,
        sep="|",
        header=None
    )

    df["origination_year"] = year

    dfs.append(df)

master_df = pd.concat(dfs, ignore_index=True)

print(master_df.shape)

Processing 2000
Processing 2001
Processing 2002
Processing 2003
Processing 2004
Processing 2005
Processing 2006
Processing 2007
Processing 2008
Processing 2009


C:\Temp\ipykernel_18252\1739466327.py:13: DtypeWarning: Columns (0: 25) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(


Processing 2010
(550000, 33)


In [4]:
column_names = ["cred_score", "first_payment_date", "first_time_homebuyer_flag", "maturity_date", 
                "metro_code_msa", "mortgage_insurance_percent", "unit_no", "occupancy_status",
                "og_cltv", "og_dti", "og_upb", "og_ltv",
                "og_int_rate", "channel", "ppm_flag", "amort_type",
                "prop_state", "prop_type", "postal_code", "loan_seq_no",
                "loan_purpose", "og_loan_term", "no_of_borrowers", "seller_name",
                "servicer_name", "super_conforming_flag", "pre_relief_refinance_loan_seq_no", "special_elig_program",
                "relief_refinance_indicator", "prop_val_method", "int_only_indicator", "MI_cal_indicator"
                ]

In [5]:
final_column_names = column_names + ['origination_year']
master_df.columns = final_column_names

In [6]:
master_df.head()

,cred_score,first_payment_date,first_time_homebuyer_flag,maturity_date,metro_code_msa,mortgage_insurance_percent,unit_no,occupancy_status,og_cltv,og_dti,...,seller_name,servicer_name,super_conforming_flag,pre_relief_refinance_loan_seq_no,special_elig_program,relief_refinance_indicator,prop_val_method,int_only_indicator,MI_cal_indicator,origination_year
0,790,200211,N,203003,NaN,30,1,P,92,29,...,Other sellers,Other servicers,NaN,NaN,9,NaN,7,N,9,2000
1,771,200011,N,201510,NaN,0,1,P,61,30,...,Other sellers,Other servicers,NaN,NaN,9,NaN,7,N,9,2000
2,762,200110,N,203003,39300.0,0,3,P,76,23,...,Other sellers,Other servicers,NaN,NaN,9,NaN,7,N,9,2000
3,737,200104,N,203004,16974.0,25,1,P,87,35,...,"NORWEST MORTGAGE, INC.","WELLS FARGO HOME MORTGAGE, INC.",NaN,NaN,9,NaN,7,N,9,2000
4,594,200003,N,203002,23460.0,0,1,P,80,24,...,Other sellers,CHASE MANHATTAN MORTGAGE CORPORATION,NaN,NaN,9,NaN,7,N,9,2000


The objective here to check if svc{year}.csv has the data of that loan for all consequent years

In [7]:
import pandas as pd
# Define column names (Freddie Mac standard svcg layout)
svcg_cols = [
    'loan_seq_no', 'monthly_period', 'current_upb', 'current_loan_delinquency_status',
    'loan_age', 'remaining_months_to_legal_maturity', 'defect_settlement_date',
    'modifications_flag', 'zero_balance_code', 'zero_balance_effective_date',
    'current_interest_rate', 'current_deferred_upb', 'due_date_last_paid_installment',
    'mi_recoveries', 'net_sales_proceeds', 'non_mi_recoveries', 'expenses',
    'legal_costs', 'maintenance_preservation_costs', 'taxes_insurance',
    'misc_expenses', 'actual_loss', 'modification_cost', 'step_modification_flag',
    'deferred_payment_modification', 'estimated_ltv', 'zero_balance_removal_upb',
    'delinquent_accrued_interest', 'delinquency_due_to_disaster',
    'borrower_assistance_status_code', 'current_period_modification_loss_amount',
    'interest_bearing_upb'
]

# Load full svcg_2003 file
svcg = pd.read_csv("C:\\Users\\Niraj Mhatre\\projects\\Mortgage-Portfolio-Risk-Analytics-and-IFRS-9-Provisioning-Framework\\Data\\sample_2003\\sample_svcg_2003.txt", sep='|', header=None, names=svcg_cols, dtype={'loan_seq_no': str, 'monthly_period': str})

# --- CHECK 1: What's the full date range?
print("Monthly period range:")
print(f"  Earliest: {svcg['monthly_period'].min()}")
print(f"  Latest:   {svcg['monthly_period'].max()}")

# --- CHECK 2: How many unique loans are tracked?
print(f"\nTotal unique loans: {svcg['loan_seq_no'].nunique():,}")
print(f"Total monthly records: {len(svcg):,}")

# --- CHECK 3: Records from 2004 onwards (loans still alive after origination year)
svcg_post2003 = svcg[svcg['monthly_period'] >= '200401']
print(f"\nRecords from Jan 2004 onwards: {len(svcg_post2003):,}")
print(f"Unique loans still active in 2004+: {svcg_post2003['loan_seq_no'].nunique():,}")

# --- CHECK 4: Year-wise breakdown of records
svcg['year'] = svcg['monthly_period'].str[:4]
print("\nRecords per year:")
print(svcg['year'].value_counts().sort_index())

# --- CHECK 5: Loans still active by year (unique loan count per year)
print("\nUnique active loans per year:")
print(svcg.groupby('year')['loan_seq_no'].nunique().sort_index())

C:\Temp\ipykernel_18252\1132666896.py:18: DtypeWarning: Columns (0: current_loan_delinquency_status, 1: modifications_flag, 2: step_modification_flag, 3: deferred_payment_modification, 4: delinquency_due_to_disaster, 5: borrower_assistance_status_code) have mixed types. Specify dtype option on import or set low_memory=False.
  svcg = pd.read_csv("C:\\Users\\Niraj Mhatre\\projects\\Mortgage-Portfolio-Risk-Analytics-and-IFRS-9-Provisioning-Framework\\Data\\sample_2003\\sample_svcg_2003.txt", sep='|', header=None, names=svcg_cols, dtype={'loan_seq_no': str, 'monthly_period': str})


Monthly period range:
  Earliest: 200301
  Latest:   202509

Total unique loans: 50,000
Total monthly records: 4,073,882

Records from Jan 2004 onwards: 3,818,439
Unique loans still active in 2004+: 48,339

Records per year:
year
2003    255443
2004    543051
2005    482995
2006    428909
2007    391596
2008    361409
2009    323416
2010    277027
2011    224769
2012    179870
2013    134704
2014    107957
2015     89730
2016     72976
2017     56750
2018     36997
2019     24314
2020     21036
2021     17330
2022     14329
2023     12012
2024     10242
2025      7020
Name: count, dtype: int64

Unique active loans per year:
year
2003    44376
2004    47839
2005    42815
2006    37382
2007    33982
2008    31334
2009    29045
2010    24764
2011    20422
2012    16701
2013    12904
2014     9747
2015     8146
2016     6711
2017     5329
2018     4018
2019     2163
2020     1867
2021     1582
2022     1289
2023     1100
2024      899
2025      807
Name: loan_seq_no, dtype: int64


Since the dataset has passed this sanity check we combine a master dataset for svc/process files

In [8]:
import pandas as pd

dfs = []

for year in range(2000, 2011):
    print(f"Processing {year}")

    file_path = (
        f"C:\\Users\\Niraj Mhatre\\projects\\Mortgage-Portfolio-Risk-Analytics-and-IFRS-9-Provisioning-Framework\\Data\\sample_{year}\\sample_svcg_{year}.txt"
    )

    df = pd.read_csv(
        file_path,
        sep="|",
        header=None
    )

    df["origination_year"] = year

    dfs.append(df)

master_monthly_df = pd.concat(dfs, ignore_index=True)

print(master_monthly_df.shape)

Processing 2000


C:\Temp\ipykernel_18252\2519369399.py:12: DtypeWarning: Columns (0: 3, 1: 7, 2: 23, 3: 24, 4: 28, 5: 29) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(


Processing 2001
Processing 2002
Processing 2003
Processing 2004
Processing 2005
Processing 2006


C:\Temp\ipykernel_18252\2519369399.py:12: DtypeWarning: Columns (0: 3, 1: 7, 2: 23, 3: 24, 4: 28) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(


Processing 2007


C:\Temp\ipykernel_18252\2519369399.py:12: DtypeWarning: Columns (0: 24, 1: 28) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(


Processing 2008
Processing 2009
Processing 2010
(33058391, 33)


renaming cols in monthly data

In [9]:
master_monthly_df.head()

,0,1,2,3,4,5,6,7,8,9,...,23,24,25,26,27,28,29,30,31,origination_year
0,F00Q10000035,200210,101000.0,0,0,329.0,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,101000.0,2000
1,F00Q10000035,200211,101000.0,0,1,328.0,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,101000.0,2000
2,F00Q10000035,200212,101000.0,0,2,327.0,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,101000.0,2000
3,F00Q10000035,200301,101000.0,0,3,326.0,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,101000.0,2000
4,F00Q10000035,200302,100000.0,0,4,325.0,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,100000.0,2000


In [10]:
master_monthly_df.shape

(33058391, 33)

In [11]:
monthly_cols = [
    "loan_seq_no",
    "monthly_reporting_period",
    "current_actual_upb",
    "current_loan_delinquency_status",
    "loan_age",
    "remaining_months_to_legal_maturity",
    "defect_settlement_date",
    "modification_flag",
    "zero_balance_code",
    "zero_balance_effective_date",
    "current_interest_rate",
    "current_non_interest_bearing_upb",
    "ddlpi",
    "mi_recoveries",
    "net_sale_proceeds",
    "non_mi_recoveries",
    "total_expenses",
    "legal_costs",
    "maintenance_and_preservation_costs",
    "taxes_and_insurance",
    "miscellaneous_expenses",
    "actual_loss_calculation",
    "cumulative_modification_cost",
    "interest_rate_step_indicator",
    "payment_deferral_flag",
    "eltv",
    "zero_balance_removal_upb",
    "delinquent_accrued_interest",
    "delinquency_due_to_disaster",
    "borrower_assistance_status_code",
    "current_month_modification_cost",
    "interest_bearing_upb",
    "origination_year"
]

In [12]:
len(monthly_cols)

33

In [13]:
master_monthly_df.columns = monthly_cols

check for null values

In [14]:

na_val = master_df.isnull().sum()
print(na_val[na_val > 0] / len(master_df))

metro_code_msa                      0.315813
super_conforming_flag               0.996125
pre_relief_refinance_loan_seq_no    0.963627
relief_refinance_indicator          0.963627
dtype: float64


imputing the values in nessecary ways

In [15]:
master_df[["metro_code_msa", "super_conforming_flag", "pre_relief_refinance_loan_seq_no", "relief_refinance_indicator"]]

,metro_code_msa,super_conforming_flag,pre_relief_refinance_loan_seq_no,relief_refinance_indicator
0,NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,NaN
2,39300.0,NaN,NaN,NaN
3,16974.0,NaN,NaN,NaN
4,23460.0,NaN,NaN,NaN
...,...,...,...,...
549995,13900.0,NaN,NaN,NaN
549996,24220.0,NaN,NaN,NaN
549997,15500.0,NaN,NaN,NaN
549998,20500.0,NaN,NaN,NaN


In [16]:
#checkimg the unique values in the columns with missing values
np.where(
    master_df["pre_relief_refinance_loan_seq_no"].notna()
)

(array([459038, 459682, 459683, ..., 549843, 549864, 549866],
       shape=(20005,)),)

In [17]:
master_df['super_conforming_flag'] = master_df['super_conforming_flag'].fillna('N')
# N Because a blank space in this dataset explicitly means "No / Standard Conforming" so we will have to use it in different ways later

In [18]:
master_df[["metro_code_msa", "pre_relief_refinance_loan_seq_no"]] = master_df[
    ["metro_code_msa", "pre_relief_refinance_loan_seq_no"]
].fillna("Unknown")

In [19]:
# Properly fill the NaNs by reassigning the column
master_df["relief_refinance_indicator"] = master_df[
    "relief_refinance_indicator"
].fillna("not applicable")

indices = np.where(master_df["relief_refinance_indicator"] == "not applicable")
print(indices)


(array([     0,      1,      2, ..., 549997, 549998, 549999],
      shape=(529995,)),)


# Imputing values for monthly data files

### first we check the prop of loans defaulted as it will help us decide what to do with the missing values

zero balance code is the code which tells if the loan is inactive along with it's reason,

🔴 Defaulted / Credit Events (Involuntary Termination)
If a loan's balance drops to zero because of a default, foreclosure, or liquidation, it will be flagged with one of these codes:

02 = Third Party Sale: The property was sold to a third party at a foreclosure auction.

03 = Short Sale or Charge Off: The property was sold for less than the remaining loan balance, or the servicer wrote off the remaining debt as a loss.

09 = REO Disposition: The foreclosure completed, Freddie Mac took ownership of the property (Real Estate Owned), and this code triggers when they finally sell/dispose of the property.

🟢 Successfully Paid Off (Voluntary Termination)
01 = Prepaid or Matured: The borrower successfully paid off the loan in full (e.g., through a refinancing, selling the home, or making the final scheduled payment).

ℹ️ Other Structural Zero Balance Codes
For completeness, the file also includes a few codes that don't neatly fall into a simple "borrower paid vs. borrower defaulted" category, usually representing administrative or portfolio changes:

15 = Whole Loan Sales: The loan was sold out of the pool to another investor.

16 = Reperforming Loan Securitizations: The loan previously delinquent became clean again and was repackaged into a new security.

96 = Defect Prior to Other Termination Event: The loan was removed from the dataset due to an eligibility or underwriting defect identified by the seller/servicer before any final payoff or default happened.

In [ ]:
# looking at proportion of each based on if they are defaulted or not or some other situation we have 3 classes here
print(master_monthly_df["current_loan_delinquency_status"].value_counts(normalize=True
        ))

In [ ]:
master_monthly_df.loc[
    master_monthly_df["zero_balance_code"].notna(),
    "zero_balance_code"
].value_counts()

KeyError: "None of [Index([nan, nan, nan, nan, nan, nan, nan, nan, nan, 1.0,\n       ...\n       nan, nan, nan, nan, nan, nan, nan, nan, nan, 1.0],\n      dtype='float64', length=33058391)] are in the [columns]"

In [49]:
arr2

,zero_balance_code
0,False
1,False
2,False
3,False
4,False
...,...
33058386,False
33058387,False
33058388,False
33058389,False


In [20]:
na_val = master_monthly_df.isnull().sum()
print(na_val[na_val > 0] / len(master_monthly_df))

remaining_months_to_legal_maturity    0.000009
defect_settlement_date                0.999892
modification_flag                     0.971975
zero_balance_code                     0.983764
zero_balance_effective_date           0.983764
ddlpi                                 0.998187
mi_recoveries                         0.999477
net_sale_proceeds                     0.999475
non_mi_recoveries                     0.999477
total_expenses                        0.999477
legal_costs                           0.999477
maintenance_and_preservation_costs    0.999477
taxes_and_insurance                   0.999477
miscellaneous_expenses                0.999477
actual_loss_calculation               0.999475
cumulative_modification_cost          0.999655
interest_rate_step_indicator          0.971975
payment_deferral_flag                 0.998118
eltv                                  0.903336
zero_balance_removal_upb              0.983764
delinquent_accrued_interest           0.999475
delinquency_d

we work in "remaining_months_to_legal_maturity" col

In [36]:
np.where(master_monthly_df["remaining_months_to_legal_maturity"].isnull())

(array([ 1389308,  1389309,  1389310,  1389311,  1389312,  1389313,
         1389314,  1389315,  1389316,  1389317,  1389318,  1389319,
         1389320,  1389321,  1389322,  1389323,  1389324,  1389325,
         1389326,  1389327,  1389328,  1389329,  1389330,  1389331,
         1389332,  1389333,  1389334,  1389335,  1389336,  1389337,
         1389338,  1389339,  1389340,  1389341,  1389342,  1389343,
         1389344,  1389345,  1389346,  1389347,  1389348,  1389349,
         1389350,  1389351,  1389352,  1389353,  1389354,  1389355,
         1389356,  1389357,  1389358,  1389359,  1389360,  1389361,
         1389362,  1389363,  1389364,  1389365,  1389366,  1389367,
         1389368,  1389369,  1389370,  1389371,  1389372,  1389373,
         1389374,  1389375,  1389376,  1389377,  1389378,  1389379,
         1389380,  1389381,  1389382,  1389383,  1389384,  1389385,
         1389386,  1389387,  1389388,  1389389,  1389390,  1389391,
         1389392,  1389393,  1389394,  1389395, 

In [42]:
master_monthly_df["remaining_months_to_legal_maturity"].isna().sum()

np.int64(304)

since the proportion is very small he idea is to drop these rows if they dont contain much info, i.e. if the rows don't have been defaulted and prop of default is less we dont need them

In [28]:
master_monthly_df[["remaining_months_to_legal_maturity","loan_seq_no"]].iloc[1389308]

remaining_months_to_legal_maturity             NaN
loan_seq_no                           F00Q40256457
Name: 1389308, dtype: object

In [29]:
master_df[master_df["loan_seq_no"] == 'F00Q40256457']

,cred_score,first_payment_date,first_time_homebuyer_flag,maturity_date,metro_code_msa,mortgage_insurance_percent,unit_no,occupancy_status,og_cltv,og_dti,...,seller_name,servicer_name,super_conforming_flag,pre_relief_refinance_loan_seq_no,special_elig_program,relief_refinance_indicator,prop_val_method,int_only_indicator,MI_cal_indicator,origination_year
48275,761,200012,N,203011,12060.0,0,1,I,80,46,...,Other sellers,Other servicers,N,Unknown,9,not applicable,7,N,9,2000


In [23]:
import pandas as pd


#def calculate_remaining_months(row):
    if pd.isna(row["remaining_months_to_legal_maturity"]):
        # Convert YYYYMM integers/strings to pandas datetime
        maturity = pd.to_datetime(str(int(row["maturity_date"])), format="%Y%m")
        reporting_month = pd.to_datetime(
            str(int(row["monthly_reporting_period"])), format="%Y%m"
        )

        # Calculate the difference in months
        return (maturity.year - reporting_month.year) * 12 + (
            maturity.month - reporting_month.month
        )
    return row["remaining_months_to_legal_maturity"]


# Apply the deterministic fix
master_monthly_df["remaining_months_to_legal_maturity"] = master_monthly_df.apply(
    calculate_remaining_months, axis=1
)

IndentationError: unexpected indent (3434941954.py, line 5)